### YOLO Dataset Processing

##### Imports & Configuration

In [11]:
import shutil
from pathlib import Path
import random

# Paths
SOURCE_DIR = Path("../data/raw/data")        # Original dataset
YOLO_DIR = Path("../data/processed/yolo_dataset")     # Target YOLO dataset folder

# Classes
CLASSES = ["need_cleaning", "free", "occupied", "awaiting"]

# Train/val/test split ratio
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

##### Create YOLO Folder Structure

In [6]:
for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print(f"YOLO folder structure created at: {YOLO_DIR.resolve()}")

YOLO folder structure created at: C:\Users\anass\Desktop\Fluxia\data\processed\yolo_dataset


##### Gather all valid images (images that have labels)

In [7]:
all_valid_images = []

for cls in CLASSES:
    img_dir = SOURCE_DIR / cls / "images"
    lbl_dir = SOURCE_DIR / cls / "labels"
    
    for img_path in img_dir.glob("*"):
        label_path = lbl_dir / img_path.with_suffix(".txt").name
        if label_path.exists() and label_path.stat().st_size > 0:
            all_valid_images.append((cls, img_path, label_path))

print(f"Total valid images: {len(all_valid_images)}")

Total valid images: 12


##### Shuffle dataset

In [9]:
random.shuffle(all_valid_images)

##### Compute split sizes

In [12]:
n_total = len(all_valid_images)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

splits = {
    "train": all_valid_images[:n_train],
    "val": all_valid_images[n_train:n_train+n_val],
    "test": all_valid_images[n_train+n_val:]
}

##### Copy images and labels into YOLO folders

In [13]:
for split_name, items in splits.items():
    for cls, img_path, label_path in items:
        new_name = f"{cls}_{img_path.name}"
        
        # Copy image
        shutil.copy(img_path, YOLO_DIR / "images" / split_name / new_name)
        # Copy label
        shutil.copy(label_path, YOLO_DIR / "labels" / split_name / new_name.replace(img_path.suffix, ".txt"))

print("Train/Val/Test split done:")
for split_name in splits.keys():
    img_count = len(list((YOLO_DIR / "images" / split_name).glob("*")))
    lbl_count = len(list((YOLO_DIR / "labels" / split_name).glob("*")))
    print(f"{split_name.upper()}: {img_count} images, {lbl_count} labels")

Train/Val/Test split done:
TRAIN: 9 images, 9 labels
VAL: 1 images, 1 labels
TEST: 2 images, 2 labels
